# Faz 5 — Yarışma Seti Üzerinde Harici Test

**Amaç:** Faz 3 (YOLOv8 2B) ve Faz 4 (nnDetection 3B) modellerini eğitim sürecinde hiç görmediğimiz **Yarışma Veri Seti** üzerinde değerlendirip karşılaştırmak. Bu adım scoping review'da işaretlediğimiz "harici doğrulama eksikliği" boşluğunu doğrudan adresliyor.

**Çıktılar**
- YOLO + nnDetection için F1 @ IoU top-5 ortalama
- Sınıf bazında precision/recall/F1
- Hasta-düzeyi multi-label AUC/sensitivity/specificity
- Karşılaştırma tablosu (2B vs 3B)

## 1. Ortam

In [ ]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT  = Path(os.environ.get("TR_ABDOMEN_BASE",  "/Users/ramazanpolat/Desktop/datasets/abdomen"))
PROJE      = Path(os.environ.get("TR_ABDOMEN_PROJE", DATA_ROOT))
CODE       = Path(os.environ.get("TR_ABDOMEN_CODE",  DATA_ROOT))
EGITIM_DIR = Path(os.environ.get("ABDOMEN_TRAIN_DIR", DATA_ROOT / "Egitim Verisi"))
TEST_DIR   = Path(os.environ.get("ABDOMEN_TEST_DIR",  DATA_ROOT / "Test Verisi"))

sys.path.insert(0, str(CODE))
from src.config import SPLIT_DIR, DET_DATA_DIR, SUPER_CLASSES, DEFAULT_WINDOWS

test_cases = sorted([d for d in TEST_DIR.iterdir() if d.is_dir()]) if TEST_DIR.exists() else []
print(f"TEST_DIR      : {TEST_DIR}  → var? {TEST_DIR.exists()}")
print(f"Test vaka sayısı: {len(test_cases)}")
print(f"İlk 5 vaka: {[d.name for d in test_cases[:5]]}")

## 2. Test PNG'leri Oluştur + Ground Truth

In [ ]:
import pandas as pd
from src.detection import export_yolo_test_dataset

# Test PNG + GT label dosyaları oluştur (manifest source='comp' satırları)
test_dir = export_yolo_test_dataset(out_root=DET_DATA_DIR)
print(f"Test dizini: {test_dir}")
print(f"  PNG: {len(list((test_dir/'images'/'test').glob('*.png')))} dosya")
print(f"  GT label: {len(list((test_dir/'labels'/'test').glob('*.txt')))} dosya")

# Ground truth DataFrame (manifest'ten — Excel'i yeniden okumaya gerek yok)
manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
comp = manifest[(manifest["source"] == "comp") &
                (manifest["bboxes"].fillna("").str.strip() != "")].copy()

gt_rows = []
for _, row in comp.iterrows():
    for token in str(row["bboxes"]).split("|"):
        parts = token.strip().split(",")
        if len(parts) != 5: continue
        try:
            sid, x1, y1, x2, y2 = (int(float(v)) for v in parts)
        except (ValueError, TypeError): continue
        if not (0 <= sid <= 5) or x2 <= x1 or y2 <= y1: continue
        gt_rows.append({"case": int(row["case"]), "image_id": int(row["image_id"]),
                        "class": sid, "x1": x1, "y1": y1, "x2": x2, "y2": y2})

gt_df = pd.DataFrame(gt_rows)
print(f"\nGT: {len(gt_df):,} bbox, {gt_df['case'].nunique()} vaka")
print(gt_df.groupby("class").size().rename(index=dict(enumerate(SUPER_CLASSES))))

## 3. YOLO 2B Test Inference

In [ ]:
from ultralytics import YOLO
from PIL import Image
from tqdm import tqdm

# Fold 0 ağırlığını bul (en iyi checkpoint)
runs_dir = PROJE / "outputs" / "runs" / "det"
weight_candidates = sorted(runs_dir.glob("fold0_*/weights/best.pt")) if runs_dir.exists() else []
YOLO_WEIGHTS = weight_candidates[-1] if weight_candidates else None
print("YOLO ağırlık:", YOLO_WEIGHTS, "→ var?", YOLO_WEIGHTS.exists() if YOLO_WEIGHTS else False)

test_img_dir = test_dir / "images" / "test"
test_imgs = sorted(test_img_dir.glob("*.png"))
print(f"Test PNG sayısı: {len(test_imgs)}")

CONF_TH = 0.05
yolo_preds = []

if YOLO_WEIGHTS and YOLO_WEIGHTS.exists():
    model = YOLO(str(YOLO_WEIGHTS))
    for ip in tqdm(test_imgs, desc="YOLO test inference"):
        stem = ip.stem
        case, image_id = (stem.split("_", 1) + ["0"])[:2]
        res = model.predict(str(ip), conf=CONF_TH, verbose=False, imgsz=512)[0]
        for box, sc, cl in zip(res.boxes.xyxy.cpu().numpy(),
                                res.boxes.conf.cpu().numpy(),
                                res.boxes.cls.cpu().numpy()):
            yolo_preds.append({"case": int(case), "image_id": int(image_id),
                               "class": int(cl), "score": float(sc),
                               "x1": float(box[0]), "y1": float(box[1]),
                               "x2": float(box[2]), "y2": float(box[3])})
    yolo_pred_df = pd.DataFrame(yolo_preds)
    log_dir = CODE / "outputs" / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    yolo_pred_df.to_csv(log_dir / "yolo_fold0_test_preds.csv", index=False)
    print(f"YOLO tahminleri: {len(yolo_pred_df):,}")
else:
    print("⚠️  YOLO ağırlığı bulunamadı — önce Faz 3'ü tamamlayın.")
    yolo_pred_df = pd.DataFrame()

## 4. nnU-Net 3B Test Inference

In [ ]:
import SimpleITK as sitk
import shutil, subprocess
from concurrent.futures import ThreadPoolExecutor

NND_ROOT      = PROJE / "outputs" / "nndet"
NIFTI_TEST    = NND_ROOT / "nifti_test"
NNUNET_RAW    = NND_ROOT / "nnunet_raw"
NNUNET_RESULTS= NND_ROOT / "nnunet_results"
DATASET_ID    = 100
DATASET_NAME  = "Abdomen"
TEST_INPUT    = NND_ROOT / "test_input"   # nnunet_raw dışında
TEST_OUTPUT   = NNUNET_RESULTS / f"Dataset{DATASET_ID}_{DATASET_NAME}" / \
                "nnUNetTrainer__nnUNetPlans__3d_fullres" / "fold_0" / "test_predictions"

for d in [NIFTI_TEST, TEST_INPUT, TEST_OUTPUT]:
    d.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"]     = str(NNUNET_RAW)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)

def _dcm2nii(case_dir):
    case = int(case_dir.name)
    out  = NIFTI_TEST / f"case_{case:05d}_0000.nii.gz"
    if out.exists(): return "skip"
    reader = sitk.ImageSeriesReader()
    names  = reader.GetGDCMSeriesFileNames(str(case_dir))
    if not names: return "no_dcm"
    reader.SetFileNames(names)
    try: sitk.WriteImage(reader.Execute(), str(out)); return "ok"
    except Exception as e: return f"err:{e}"

with ThreadPoolExecutor(max_workers=4) as ex:
    res = list(tqdm(ex.map(_dcm2nii, test_cases), total=len(test_cases), desc="Test DICOM→NIfTI"))
print(f"NIfTI: {res.count('ok')} yeni, {res.count('skip')} atlandı")

# Test görüntülerini predict input dizinine semlink
for c in test_cases:
    case = int(c.name)
    src = NIFTI_TEST / f"case_{case:05d}_0000.nii.gz"
    dst = TEST_INPUT  / f"case_{case:05d}_0000.nii.gz"
    if src.exists() and not dst.exists():
        try: os.symlink(src, dst)
        except (OSError, NotImplementedError): shutil.copy2(src, dst)
print(f"TEST_INPUT: {len(list(TEST_INPUT.glob('*.nii.gz')))} dosya")

In [ ]:
import shutil, sysconfig
import numpy as np
from scipy import ndimage

def _nnunet_cmd(name):
    p = shutil.which(name)
    if p: return p
    for d in [sysconfig.get_path("scripts"), str(Path(sys.executable).parent)]:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name

_cmd = _nnunet_cmd("nnUNetv2_predict")
print(f"nnUNetv2_predict: {_cmd}")
r = subprocess.run([
    _cmd,
    "-i", str(TEST_INPUT),
    "-o", str(TEST_OUTPUT),
    "-d", str(DATASET_ID),
    "-c", "3d_fullres",
    "-f", "0",
    "--save_probabilities",
], env={**os.environ}, capture_output=True, text=True, timeout=3600*6)

if r.returncode == 0:
    preds = list(TEST_OUTPUT.glob("*.nii.gz"))
    print(f"✓ Tahmin tamamlandı: {len(preds)} NIfTI mask")
else:
    print("STDOUT:", r.stdout[-1000:])
    print("STDERR:", r.stderr[-500:])

In [ ]:
def seg_to_bboxes_2d(pred_nii_path):
    """nnU-Net semantic mask → 2D projeksiyon bbox satırları."""
    try: case = int(pred_nii_path.stem.split("_")[1])
    except (IndexError, ValueError): return []
    mask = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_nii_path)))  # [Z,Y,X]
    prob_path = pred_nii_path.with_suffix("").with_suffix(".npz")
    probs = np.load(prob_path)["probabilities"] if prob_path.exists() else None
    rows = []
    for label_id in range(1, len(SUPER_CLASSES) + 1):
        binary = (mask == label_id).astype(np.uint8)
        if binary.sum() == 0: continue
        labeled, n_comp = ndimage.label(binary)
        total_vox = float(binary.sum())
        for comp_id in range(1, n_comp + 1):
            comp_mask = (labeled == comp_id)
            coords = np.where(comp_mask)
            z1, z2 = int(coords[0].min()), int(coords[0].max())
            y1, y2 = int(coords[1].min()), int(coords[1].max())
            x1, x2 = int(coords[2].min()), int(coords[2].max())
            score = float(probs[label_id][comp_mask].mean()) if probs is not None \
                    else float(comp_mask.sum()) / total_vox
            for z in range(z1, z2 + 1):
                rows.append({"case": case, "image_id": z, "class": label_id - 1,
                             "score": score,
                             "x1": float(x1), "y1": float(y1),
                             "x2": float(x2), "y2": float(y2)})
    return rows

all_rows = []
for p in sorted(TEST_OUTPUT.glob("*.nii.gz")):
    all_rows.extend(seg_to_bboxes_2d(p))

nnunet_pred_df = pd.DataFrame(all_rows)
print(f"nnU-Net tahminleri (2D proj): {len(nnunet_pred_df):,} bbox, "
      f"{nnunet_pred_df['case'].nunique() if not nnunet_pred_df.empty else 0} vaka")
if not nnunet_pred_df.empty:
    log_dir = CODE / "outputs" / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    nnunet_pred_df.to_csv(log_dir / "nnunet_fold0_test_preds.csv", index=False)

## 5. Karşılaştırmalı Değerlendirme

In [ ]:
from src.evaluation import top5_f1_mean, f1_at_iou

def evaluate(name, pred_df):
    if pred_df.empty:
        return {"model": name, "n_pred": 0, "top5_f1": None,
                "macro_f1@0.3": None, "macro_f1@0.5": None}
    top5 = top5_f1_mean(pred_df, gt_df)
    d3   = f1_at_iou(pred_df, gt_df, 0.3)
    d5   = f1_at_iou(pred_df, gt_df, 0.5)
    return {
        "model": name,
        "n_pred": len(pred_df),
        "top5_f1": round(top5["top5_mean_f1"], 4),
        "macro_f1@0.3": round(d3["macro_f1"], 4),
        "macro_f1@0.5": round(d5["macro_f1"], 4),
        "micro_f1@0.3": round(d3["micro_f1"], 4),
    }

summary = pd.DataFrame([
    evaluate("YOLOv8 2B (Faz 3)", yolo_pred_df),
    evaluate("nnU-Net 3B (Faz 4)", nnunet_pred_df),
])
print(summary.to_string(index=False))

In [ ]:
rows = []
for mn, pred_df in [("YOLOv8_2B", yolo_pred_df), ("nnUNet_3B", nnunet_pred_df)]:
    if pred_df.empty: continue
    d = f1_at_iou(pred_df, gt_df, 0.3)
    for cid, m in d["per_class"].items():
        if cid >= len(SUPER_CLASSES): continue
        rows.append({"model": mn, "class": SUPER_CLASSES[cid],
                     "precision": round(m["precision"], 3),
                     "recall":    round(m["recall"], 3),
                     "f1":        round(m["f1"], 3),
                     "TP": m["tp"], "FP": m["fp"], "FN": m["fn"]})

if rows:
    class_table = pd.DataFrame(rows)
    print(class_table.pivot_table(index="class", columns="model",
                                  values=["precision", "recall", "f1"],
                                  aggfunc="first").round(3))

## 6. Hasta-Düzeyi Multi-Label

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

def patient_level(pred_df, gt_df):
    all_cases = sorted(set(pred_df['case']).union(set(gt_df['case'])))
    n_cls = len(SUPER_CLASSES)
    y_true = np.zeros((len(all_cases), n_cls), dtype=int)
    y_pred = np.zeros((len(all_cases), n_cls), dtype=int)
    y_score = np.zeros((len(all_cases), n_cls), dtype=float)
    idx = {c: i for i, c in enumerate(all_cases)}
    for _, r in gt_df.iterrows():
        i = idx.get(int(r['case']))
        if i is None or int(r['class']) >= n_cls: continue
        y_true[i, int(r['class'])] = 1
    if not pred_df.empty:
        for case, grp in pred_df.groupby('case'):
            i = idx.get(int(case))
            if i is None: continue
            for cls, sub in grp.groupby('class'):
                if int(cls) >= n_cls: continue
                y_pred[i, int(cls)] = 1
                y_score[i, int(cls)] = float(sub['score'].max())
    return y_true, y_pred, y_score, all_cases

for mn, pd_df in [("YOLO_2B", yolo_pred_df), ("nnDet_3B", nnunet_pred_df)]:
    if pd_df.empty: continue
    yt, yp, ys, _ = patient_level(pd_df, gt_df)
    print(f"\n=== {mn} — Hasta düzeyi ===")
    for cid in range(len(SUPER_CLASSES)):
        if yt[:, cid].sum() < 2: continue
        try: auc = roc_auc_score(yt[:, cid], ys[:, cid])
        except: auc = float('nan')
        sens = yp[yt[:, cid]==1, cid].mean() if yt[:, cid].sum() else 0
        spec = 1 - yp[yt[:, cid]==0, cid].mean() if (yt[:, cid]==0).sum() else 0
        print(f"  {SUPER_CLASSES[cid]:30s} AUC={auc:.3f}  Sens={sens:.3f}  Spec={spec:.3f}  "
              f"(n_pos={int(yt[:,cid].sum())})")

## 7. Çıktı Kaydet

In [ ]:
log_dir = CODE / "outputs" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)
out = log_dir / "test_karsilastirma.json"
out.write_text(json.dumps({
    "summary": summary.to_dict(orient="records"),
    "n_test_cases": len(test_cases),
    "n_gt_bboxes": len(gt_df),
}, indent=2, ensure_ascii=False))
print("Kayıt:", out)
print("\nÖZET:")
print(summary.to_string(index=False))

## 8. Faz 5 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| YOLO yarışma tahminleri | `outputs/logs/yolo_fold0_yarisma_preds.csv` |
| nnDetection yarışma tahminleri | `outputs/logs/nndet_fold0_yarisma_preds.csv` |
| Karşılaştırma raporu | `outputs/logs/yarisma_karsilastirma.json` |

**Makale kritik tablosu:** YOLO 2B vs nnDetection 3B (top-5 F1, macro F1@0.3, macro F1@0.5) + hasta-düzeyi AUC/sens/spec.

**Sonraki:** `Faz6_Ablation_HU_Pencereleri.ipynb`